In [1]:
!pip install transformers accelerate soundfile langdetect torchaudio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 15.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=f3595dc60d2e264daafd9ade7d755f4e61ab19629ba40f7270825db9a9d730c3
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
#!/usr/bin/env python3
"""Task 1.1: Frame-level English/Hindi LID with self-supervised bootstrapping.
"""

from __future__ import annotations

import argparse
import gc
import json
import random
import re
from dataclasses import dataclass
from pathlib import Path
from typing import List, Optional, Sequence, Tuple

import numpy as np
import soundfile as sf
import torch
import torch.nn as nn
from langdetect import DetectorFactory, LangDetectException, detect
from torch.utils.data import DataLoader, TensorDataset
from transformers import (
    AutoFeatureExtractor,
    AutoProcessor,
    Wav2Vec2Model,
    WhisperForConditionalGeneration,
    WhisperProcessor,
)

In [ ]:
# ──────────────────── constants ────────────────────
TARGET_SR = 16_000
ENGLISH = 0
HINDI = 1
IDX_TO_LANG = {ENGLISH: "English", HINDI: "Hindi"}

# ──────────────────── data structures ──────────────
@dataclass
class WordInterval:
    start_s: float
    end_s: float
    text: str
    label: int


# ──────────────────── classifier head ──────────────
class LIDHead(nn.Module):
    """Two-layer frame-level classification head (768-dim input for wav2vec2-base)."""

    def __init__(self, input_dim: int = 768, hidden_dim: int = 256, out_dim: int = 2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


# ──────────────────── utilities ────────────────────
def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    DetectorFactory.seed = seed


def resolve_audio_path(path_str: str) -> Path:
    path = Path(path_str)
    if path.exists():
        return path
    fallback = Path("input") / path_str
    if fallback.exists():
        return fallback
    raise FileNotFoundError(
        f"Audio file '{path_str}' not found! Please make sure you have uploaded "
        f"'{path_str}' directly to the current Colab directory."
    )


def load_mono_16k(path: Path) -> torch.Tensor:
    """Load audio file, convert to mono 16 kHz float32 tensor."""
    wav_np, sr = sf.read(str(path), always_2d=True, dtype="float32")
    if wav_np.size == 0:
        raise ValueError(f"Empty audio file: {path}")

    wav = torch.from_numpy(wav_np.T)  # [channels, samples]
    if wav.size(0) > 1:
        wav = wav.mean(dim=0, keepdim=True)

    if sr != TARGET_SR:
        wav = torch.nn.functional.interpolate(
            wav.unsqueeze(0),
            size=int(round(wav.size(1) * TARGET_SR / sr)),
            mode="linear",
            align_corners=False,
        ).squeeze(0)

    return wav.squeeze(0).contiguous()


def clean_word(text: str) -> str:
    return re.sub(r"[^A-Za-z\u0900-\u097F]+", "", text).strip()


def infer_word_lang(word: str) -> Optional[int]:
    token = clean_word(word)
    if not token:
        return None

    # Devanagari characters → Hindi
    if re.search(r"[\u0900-\u097F]", token):
        return HINDI

    try:
        lang = detect(token)
    except LangDetectException:
        return None

    if lang == "hi":
        return HINDI
    return ENGLISH


# ──────────────── Whisper pseudo-labelling ─────────
def run_whisper_word_timestamps(
    audio_path: Path,
    device: torch.device,
    model_id: str = "openai/whisper-large-v3",
    chunk_length_s: float = 30.0,
) -> List[Tuple[float, float, str]]:
    """Run Whisper ASR on CPU in explicit chunks to limit peak memory.

    Instead of using the HF pipeline (which loads the entire audio and can OOM),
    we manually split the audio into chunks, transcribe each with word timestamps
    via model.generate(), and stitch results.
    """
    print(f"  Loading Whisper model: {model_id} on {device}...")
    processor = WhisperProcessor.from_pretrained(model_id)
    model = WhisperForConditionalGeneration.from_pretrained(
        model_id, torch_dtype=torch.float16 if device.type == "cuda" else torch.float32,
    ).to(device)
    model.eval()

    # Load full audio
    wav_np, sr = sf.read(str(audio_path), dtype="float32")
    if wav_np.ndim > 1:
        wav_np = wav_np.mean(axis=1)
    if sr != TARGET_SR:
        # Simple linear resample
        n_out = int(len(wav_np) * TARGET_SR / sr)
        wav_np = np.interp(
            np.linspace(0, len(wav_np) - 1, n_out),
            np.arange(len(wav_np)),
            wav_np,
        ).astype(np.float32)
        sr = TARGET_SR

    total_samples = len(wav_np)
    chunk_samples = int(chunk_length_s * sr)
    words: List[Tuple[float, float, str]] = []

    print(f"  Running ASR on {total_samples/sr:.1f}s audio in {chunk_length_s}s chunks...")
    n_chunks = (total_samples + chunk_samples - 1) // chunk_samples

    for ci in range(n_chunks):
        start_sample = ci * chunk_samples
        end_sample = min(total_samples, start_sample + chunk_samples)
        chunk_audio = wav_np[start_sample:end_sample]
        chunk_offset_s = start_sample / sr

        if len(chunk_audio) < 400:
            continue

        # Process with Whisper
        inputs = processor(
            chunk_audio, sampling_rate=sr, return_tensors="pt",
        )
        input_features = inputs.input_features.to(
            device, dtype=torch.float16 if device.type == "cuda" else torch.float32
        )

        with torch.no_grad():
            generated = model.generate(
                input_features,
                return_timestamps=True,
                task="transcribe",
            )

        # Decode with timestamp tokens preserved
        raw_text = processor.tokenizer.decode(generated[0], decode_with_timestamps=True)
        # Parse word timestamps from the decoded output
        chunk_words = _parse_whisper_timestamps(raw_text, chunk_offset_s, chunk_length_s)
        words.extend(chunk_words)

        if (ci + 1) % 5 == 0 or ci == n_chunks - 1:
            print(f"    ASR chunk {ci + 1}/{n_chunks} done ({len(words)} words so far)")
        gc.collect()

    del model, processor
    gc.collect()
    print(f"  Whisper unloaded. Total words extracted: {len(words)}")
    return words


def _parse_whisper_timestamps(
    decoded_text: str,
    offset_s: float,
    max_chunk_s: float,
) -> List[Tuple[float, float, str]]:
    """Parse Whisper decoded output with timestamp tokens into word intervals.

    Whisper output format: <|0.00|> word word <|2.40|> word <|3.60|> ...
    """
    # Find all timestamp tokens and text between them
    pattern = r'<\|(\d+\.\d+)\|>'
    tokens = re.split(pattern, decoded_text)

    words: List[Tuple[float, float, str]] = []
    i = 0
    while i < len(tokens):
        if _is_float(tokens[i]):
            start_ts = float(tokens[i])
            if i + 1 < len(tokens):
                text = re.sub(r'<\|[^>]+\|>', '', tokens[i + 1]).strip()
                if i + 2 < len(tokens) and _is_float(tokens[i + 2]):
                    end_ts = float(tokens[i + 2])
                    if text and end_ts > start_ts:
                        abs_start = offset_s + min(start_ts, max_chunk_s)
                        abs_end = offset_s + min(end_ts, max_chunk_s)
                        if abs_end > abs_start:
                            word_list = text.split()
                            if word_list:
                                word_dur = (abs_end - abs_start) / len(word_list)
                                for wi, w in enumerate(word_list):
                                    ws = abs_start + wi * word_dur
                                    we = ws + word_dur
                                    words.append((ws, we, w))
                    i += 2
                    continue
        i += 1

    return words


def _is_float(s: str) -> bool:
    try:
        float(s)
        return True
    except (ValueError, TypeError):
        return False


def build_word_intervals(words: Sequence[Tuple[float, float, str]]) -> List[WordInterval]:
    intervals: List[WordInterval] = []
    for start_s, end_s, text in words:
        label = infer_word_lang(text)
        if label is None:
            continue
        intervals.append(WordInterval(start_s=start_s, end_s=end_s, text=text, label=label))

    intervals.sort(key=lambda x: x.start_s)
    return intervals


# ──────────── wav2vec2 frame feature extraction ────
def extract_frame_features(
    waveform: torch.Tensor,
    feature_extractor,
    wav2vec: Wav2Vec2Model,
    device: torch.device,
    chunk_s: float = 30.0,
) -> Tuple[torch.Tensor, torch.Tensor]:
    """Extract frame-level features in chunks to limit peak memory."""
    chunk_samples = int(chunk_s * TARGET_SR)
    total_samples = waveform.numel()
    if total_samples == 0:
        raise ValueError("Input waveform is empty.")

    feature_chunks: List[torch.Tensor] = []
    time_chunks: List[torch.Tensor] = []

    n_chunks = (total_samples + chunk_samples - 1) // chunk_samples
    with torch.no_grad():
        for ci, start in enumerate(range(0, total_samples, chunk_samples)):
            end = min(total_samples, start + chunk_samples)
            chunk = waveform[start:end]
            if chunk.numel() < 400:
                pad_len = 400 - chunk.numel()
                chunk = torch.nn.functional.pad(chunk, (0, pad_len))

            model_inputs = feature_extractor(
                chunk.numpy(),
                sampling_rate=TARGET_SR,
                return_tensors="pt",
            )
            input_values = model_inputs.input_values.to(device)
            attention_mask = (
                model_inputs.attention_mask.to(device)
                if hasattr(model_inputs, "attention_mask")
                   and model_inputs.attention_mask is not None
                else None
            )

            out = wav2vec(input_values=input_values, attention_mask=attention_mask)
            hidden = out.last_hidden_state.squeeze(0).cpu()  # [frames, 768]
            n_frames = hidden.size(0)

            chunk_duration = (end - start) / float(TARGET_SR)
            start_s = start / float(TARGET_SR)
            frame_idx = torch.arange(n_frames, dtype=torch.float32)
            frame_times = start_s + ((frame_idx + 0.5) / max(n_frames, 1)) * chunk_duration

            feature_chunks.append(hidden)
            time_chunks.append(frame_times)

            if (ci + 1) % 5 == 0 or ci == n_chunks - 1:
                print(f"    chunk {ci + 1}/{n_chunks} done")
            gc.collect()

    features = torch.cat(feature_chunks, dim=0)
    frame_times = torch.cat(time_chunks, dim=0)
    return features, frame_times


# ──────────── label assignment & metrics ───────────
def assign_frame_labels(
    frame_times: torch.Tensor,
    intervals: Sequence[WordInterval],
) -> torch.Tensor:
    labels = torch.full((frame_times.numel(),), -100, dtype=torch.long)
    if not intervals:
        return labels

    idx = 0
    n_intervals = len(intervals)
    for i, t in enumerate(frame_times.tolist()):
        while idx < n_intervals and intervals[idx].end_s <= t:
            idx += 1
        if idx >= n_intervals:
            break
        cur = intervals[idx]
        if cur.start_s <= t < cur.end_s:
            labels[i] = cur.label

    return labels


def macro_f1_score(y_true: torch.Tensor, y_pred: torch.Tensor) -> float:
    eps = 1e-12
    f1_values = []
    for cls in (ENGLISH, HINDI):
        tp = torch.sum((y_true == cls) & (y_pred == cls)).item()
        fp = torch.sum((y_true != cls) & (y_pred == cls)).item()
        fn = torch.sum((y_true == cls) & (y_pred != cls)).item()
        precision = tp / (tp + fp + eps)
        recall = tp / (tp + fn + eps)
        f1 = 2.0 * precision * recall / (precision + recall + eps)
        f1_values.append(f1)
    return float(sum(f1_values) / len(f1_values))


# ──────────── training ─────────────────────────────
def train_lid_head(
    features: torch.Tensor,
    labels: torch.Tensor,
    device: torch.device,
    epochs: int,
    batch_size: int,
    lr: float,
) -> Tuple[LIDHead, float]:
    valid_mask = labels >= 0
    x = features[valid_mask]
    y = labels[valid_mask]
    if x.size(0) < 20:
        raise ValueError("Too few pseudo-labeled frames for training.")

    perm = torch.randperm(x.size(0))
    x = x[perm]
    y = y[perm]

    n_val = max(1, int(0.1 * x.size(0)))
    n_train = x.size(0) - n_val
    x_train, y_train = x[:n_train], y[:n_train]
    x_val, y_val = x[n_train:], y[n_train:]

    model = LIDHead().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    train_loader = DataLoader(
        TensorDataset(x_train, y_train), batch_size=batch_size, shuffle=True
    )

    final_val_f1 = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        total_count = 0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = model(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()

            bs = yb.size(0)
            total_loss += float(loss.item()) * bs
            total_count += bs

        model.eval()
        with torch.no_grad():
            x_val_d, y_val_d = x_val.to(device), y_val.to(device)
            val_logits = model(x_val_d)
            val_pred = torch.argmax(val_logits, dim=-1)
            final_val_f1 = macro_f1_score(y_val_d, val_pred)

        avg_loss = total_loss / max(total_count, 1)
        print(f"  Epoch {epoch:02d}/{epochs} | loss={avg_loss:.4f} | val_macro_f1={final_val_f1:.4f}")

    return model, final_val_f1


# ──────────── post-processing ──────────────────────
def median_filter_labels(labels: torch.Tensor, width: int = 5) -> torch.Tensor:
    if width < 1 or width % 2 == 0:
        raise ValueError("Median filter width must be an odd positive integer.")
    if labels.numel() == 0:
        return labels

    radius = width // 2
    out = labels.clone()
    for i in range(labels.numel()):
        lo = max(0, i - radius)
        hi = min(labels.numel(), i + radius + 1)
        window = labels[lo:hi]
        counts = torch.bincount(window, minlength=2)
        out[i] = torch.argmax(counts)
    return out


def apply_transition_penalty(
    labels: torch.Tensor,
    frame_times: torch.Tensor,
    min_switch_ms: float = 200.0,
    max_passes: int = 5,
) -> torch.Tensor:
    """Merge short A-B-A switches where B is shorter than min_switch_ms."""
    if labels.numel() == 0:
        return labels

    out = labels.clone()
    for _ in range(max_passes):
        starts, ends = frame_boundaries_sec(frame_times)

        runs: List[Tuple[int, int, int]] = []
        run_start = 0
        for i in range(1, out.numel()):
            if int(out[i].item()) != int(out[i - 1].item()):
                runs.append((run_start, i - 1, int(out[i - 1].item())))
                run_start = i
        runs.append((run_start, out.numel() - 1, int(out[-1].item())))

        if len(runs) < 3:
            break

        changed = False
        for j in range(1, len(runs) - 1):
            s, e, cur_label = runs[j]
            _, _, prev_label = runs[j - 1]
            _, _, next_label = runs[j + 1]

            if prev_label != next_label or cur_label == prev_label:
                continue

            dur_ms = max(0.0, float((ends[e] - starts[s]).item()) * 1000.0)
            if dur_ms < min_switch_ms:
                out[s : e + 1] = prev_label
                changed = True

        if not changed:
            break

    return out


def frame_boundaries_sec(frame_times: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor]:
    n = frame_times.numel()
    if n == 0:
        return torch.empty(0), torch.empty(0)

    if n == 1:
        dt = 0.02
        return torch.tensor([max(0.0, float(frame_times[0]) - dt / 2)]), torch.tensor(
            [float(frame_times[0]) + dt / 2]
        )

    mids = 0.5 * (frame_times[:-1] + frame_times[1:])
    starts = torch.empty_like(frame_times)
    ends = torch.empty_like(frame_times)

    starts[0] = max(0.0, float(frame_times[0] - (mids[0] - frame_times[0])))
    starts[1:] = mids
    ends[:-1] = mids
    ends[-1] = frame_times[-1] + (frame_times[-1] - mids[-1])
    return starts, ends


def labels_to_segments(
    frame_times: torch.Tensor,
    labels: torch.Tensor,
    confidences: torch.Tensor,
) -> List[dict]:
    starts, ends = frame_boundaries_sec(frame_times)
    if labels.numel() == 0:
        return []

    segments: List[dict] = []
    cur_label = int(labels[0].item())
    cur_start = float(starts[0].item())
    cur_end = float(ends[0].item())
    cur_conf = [float(confidences[0].item())]

    for i in range(1, labels.numel()):
        lbl = int(labels[i].item())
        st = float(starts[i].item())
        en = float(ends[i].item())
        cf = float(confidences[i].item())

        if lbl == cur_label and st <= cur_end + 1e-6:
            cur_end = en
            cur_conf.append(cf)
            continue

        segments.append(
            {
                "start_ms": int(round(cur_start * 1000.0)),
                "end_ms": int(round(cur_end * 1000.0)),
                "language": IDX_TO_LANG[cur_label],
                "confidence": round(float(sum(cur_conf) / max(len(cur_conf), 1)), 4),
            }
        )

        cur_label = lbl
        cur_start = st
        cur_end = en
        cur_conf = [cf]

    segments.append(
        {
            "start_ms": int(round(cur_start * 1000.0)),
            "end_ms": int(round(cur_end * 1000.0)),
            "language": IDX_TO_LANG[cur_label],
            "confidence": round(float(sum(cur_conf) / max(len(cur_conf), 1)), 4),
        }
    )
    return segments


# ──────────── CLI ──────────────────────────────────
def parse_args() -> argparse.Namespace:
    parser = argparse.ArgumentParser(
        description="Task 1.1 frame-level Hindi/English LID (Colab GPU-optimised)"
    )
    parser.add_argument(
        "--bootstrap-audio",
        default="original_segment.wav",
        help="Audio used for Whisper pseudo-label bootstrapping.",
    )
    parser.add_argument(
        "--inference-audio",
        default="denoised_segment.wav",
        help="Audio used for final LID inference output.",
    )
    parser.add_argument(
        "--output-json",
        default="lid_labels.json",
        help="Output JSON path.",
    )
    parser.add_argument(
        "--whisper-model",
        default="openai/whisper-large-v3",
        help="Whisper model id for bootstrapping.",
    )
    parser.add_argument(
        "--wav2vec-model",
        default="facebook/wav2vec2-base",
        help="Wav2Vec2 model id for frozen feature extraction.",
    )
    parser.add_argument("--epochs", type=int, default=10, help="Training epochs.")
    parser.add_argument("--batch-size", type=int, default=512, help="Training batch size.")
    parser.add_argument("--lr", type=float, default=1e-3, help="Adam learning rate.")
    parser.add_argument("--seed", type=int, default=42, help="Random seed.")
    parser.add_argument(
        "--chunk-seconds",
        type=float,
        default=30.0,
        help="Chunk length in seconds for processing (keeps memory low).",
    )
    parser.add_argument(
        "--min-switch-ms",
        type=float,
        default=200.0,
        help="Merge A-B-A language switches where B is shorter than this in ms.",
    )
    parser.add_argument(
        "--save-model",
        default="lid_model.pt",
        help="Path to save the trained LID head weights (for reuse in Task 4.2).",
    )
    return parser.parse_args()


# ──────────── main ─────────────────────────────────\n

In [6]:

class Args: pass
args = Args()
args.bootstrap_audio = "original_segment.wav"
args.inference_audio = "denoised_segment.wav"
args.audio = "original_segment.wav" # for task 1.2
args.corpus = "syllabus_corpus.txt"
args.output = "transcript_constrained.json"
args.output_json = "lid_labels.json"
args.whisper_model = "openai/whisper-large-v3"
args.wav2vec_model = "facebook/wav2vec2-base"
args.ngram_order = 3
args.bias_weight = 2.5
args.epochs = 10
args.batch_size = 512
args.lr = 1e-3
args.seed = 42
args.chunk_seconds = 30.0
args.min_switch_ms = 200.0
args.save_model = "lid_model.pt"

set_seed(args.seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

bootstrap_audio = resolve_audio_path(args.bootstrap_audio)
print(f"Bootstrap audio: {bootstrap_audio}")

# ── Step 1: Whisper pseudo-labels (then free memory) ──

Using device: cuda
Bootstrap audio: original_segment.wav


In [8]:
print("\n[Step 1/5] Running Whisper for pseudo word timestamps...")
whisper_words = run_whisper_word_timestamps(
    audio_path=bootstrap_audio,
    device=device,
    model_id=args.whisper_model,
    chunk_length_s=args.chunk_seconds,
)
intervals = build_word_intervals(whisper_words)
if not intervals:
    raise RuntimeError(
        "No pseudo-labeled intervals generated from Whisper + langdetect."
    )

eng_count = sum(1 for iv in intervals if iv.label == ENGLISH)
hin_count = sum(1 for iv in intervals if iv.label == HINDI)
print(f"  Pseudo-labeled words: {len(intervals)} (English={eng_count}, Hindi={hin_count})")

# ── Step 2: wav2vec2 frame features for training ──


[Step 1/5] Running Whisper for pseudo word timestamps...
  Loading Whisper model: openai/whisper-large-v3 on cuda...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

  Running ASR on 599.6s audio in 30.0s chunks...


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensA

    ASR chunk 5/20 done (239 words so far)
    ASR chunk 10/20 done (541 words so far)
    ASR chunk 15/20 done (778 words so far)
    ASR chunk 20/20 done (1007 words so far)
  Whisper unloaded. Total words extracted: 1007
  Pseudo-labeled words: 1006 (English=439, Hindi=567)


In [9]:
print("\n[Step 2/5] Extracting frame features for training...")
feature_extractor = AutoFeatureExtractor.from_pretrained(args.wav2vec_model)
wav2vec = Wav2Vec2Model.from_pretrained(args.wav2vec_model).to(device)
wav2vec.eval()
for p in wav2vec.parameters():
    p.requires_grad = False

train_wave = load_mono_16k(bootstrap_audio)
train_features, train_times = extract_frame_features(
    train_wave, feature_extractor, wav2vec, device, chunk_s=args.chunk_seconds
)
train_labels = assign_frame_labels(train_times, intervals)
n_labeled = int((train_labels >= 0).sum().item())
print(f"  Labeled frames: {n_labeled} / {train_labels.numel()}")

# ── Step 3: Train LID head ──


[Step 2/5] Extracting frame features for training...


preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


    chunk 5/20 done
    chunk 10/20 done
    chunk 15/20 done
    chunk 20/20 done
  Labeled frames: 24289 / 29959


In [ ]:
print("\n[Step 3/5] Training LID head...")
lid_head = LIDHead().to(device)
lid_head, val_f1 = train_lid_head(
    train_features,
    train_labels,
    device,
    epochs=args.epochs,
    batch_size=args.batch_size,
    lr=args.lr,
)
print(f"  Held-out frame-level macro F1 (10% split): {val_f1:.4f}")

# Save model for reuse in Task 4.2
torch.save(lid_head.state_dict(), args.save_model)
print(f"  Saved LID head weights to: {args.save_model}")

# ── Step 4: Inference on denoised audio ──
try:
    inference_audio = resolve_audio_path(args.inference_audio)
except FileNotFoundError:
    print(f"\n  Warning: '{args.inference_audio}' not found, using bootstrap audio for inference.")
    inference_audio = bootstrap_audio

print(f"\n[Step 4/5] Running frame-level inference on: {inference_audio}")
infer_wave = load_mono_16k(inference_audio)
infer_features, infer_times = extract_frame_features(
    infer_wave, feature_extractor, wav2vec, device, chunk_s=args.chunk_seconds
)

# Free wav2vec2 from memory
del wav2vec, feature_extractor
gc.collect()

lid_head.eval()
with torch.no_grad():
    infer_features_d = infer_features.to(device)
    logits = lid_head(infer_features_d).cpu()
    probs = torch.softmax(logits, dim=-1)
    labels = torch.argmax(probs, dim=-1)

# ── Step 5: Post-processing & save ──


[Step 3/5] Training LID head...
  Epoch 01/10 | loss=0.3691 | val_macro_f1=0.9658
  Epoch 02/10 | loss=0.1132 | val_macro_f1=0.9893
  Epoch 03/10 | loss=0.0613 | val_macro_f1=0.9934
  Epoch 04/10 | loss=0.0432 | val_macro_f1=0.9967
  Epoch 05/10 | loss=0.0314 | val_macro_f1=0.9971
  Epoch 06/10 | loss=0.0248 | val_macro_f1=0.9984
  Epoch 07/10 | loss=0.0190 | val_macro_f1=0.9988
  Epoch 08/10 | loss=0.0159 | val_macro_f1=0.9992
  Epoch 09/10 | loss=0.0123 | val_macro_f1=0.9992
  Epoch 10/10 | loss=0.0105 | val_macro_f1=0.9984
  Held-out frame-level macro F1 (10% split): 0.9984
  Saved LID head weights to: lid_model.pt

[Step 4/5] Running frame-level inference on: denoised_segment.wav
    chunk 5/20 done
    chunk 10/20 done
    chunk 15/20 done
    chunk 20/20 done


In [11]:
print("\n[Step 5/5] Post-processing (median filter + transition penalty)...")
smooth_labels = median_filter_labels(labels, width=5)
smooth_labels = apply_transition_penalty(
    smooth_labels,
    infer_times,
    min_switch_ms=args.min_switch_ms,
)
confidence = probs[torch.arange(probs.size(0)), smooth_labels]
segments = labels_to_segments(infer_times, smooth_labels, confidence)

with open(args.output_json, "w", encoding="utf-8") as f:
    json.dump(segments, f, indent=2)

print(f"\n  Saved {len(segments)} language segments to: {args.output_json}")
print("  Done!")



[Step 5/5] Post-processing (median filter + transition penalty)...

  Saved 53 language segments to: lid_labels.json
  Done!
